# JEPA and the Platonic theorem — the arithmetic behind the post

Companion notebook for the 2026-11-04 post on Balestriero & LeCun, arXiv 2610.39302. All headline numbers are the paper's reported figures; the rotation demo is a small toy that checks the theorem's core claim numerically.

In [ ]:
# Headline arithmetic
ratio = 72e9 / 109e6
print(f"72B / 109M = {ratio:.1f}x  (the '660x' headline)")
print(f"alignment gain over Qwen2.5-72B: {0.184-0.154:+.3f} abs, {(0.184/0.154-1)*100:.1f}% rel")
print(f"alignment gain over CLIP text: {0.184-0.124:+.3f} abs, {(0.184/0.124-1)*100:.1f}% rel")
print(f"sample ratio: {900/21.6:.1f}x  (900M / 21.6M)")
print(f"ImageNet-100 pair-free gap vs rerun best: {64.4-47.6:.1f} pp")
assert 650 < ratio < 670 and abs(900/21.6 - 41.7) < 0.1

## Toy check: two embeddings that differ by one rotation coincide after Procrustes alignment

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
world = rng.normal(size=(200, 16))          # 200 'world states' in 16 dims
theta = 0.7
# random orthogonal rotation via QR
Q, _ = np.linalg.qr(rng.normal(size=(16, 16)))
emb_a = world
emb_b = world @ Q.T                          # model B = model A, rotated
# Procrustes: recover the rotation from paired points
U, _, Vt = np.linalg.svd(emb_a.T @ emb_b)
R_hat = U @ Vt
err = np.linalg.norm(emb_b @ R_hat.T - emb_a) / np.linalg.norm(emb_a)
print(f"relative error after undoing the rotation: {err:.2e}")
assert err < 1e-10, "rotation recovery failed"
print("OK: same world, one rotation apart -> identical after alignment")

## Toy check: good views isolate the world, bad views keep the modality's noise

Each observation = world state + a modality-specific nuisance (camera angle, wording, sensor noise). A representation built by averaging two views: with good views the independent nuisances partly cancel, so the world share of variance rises; with bad views the second view is a corruption of the first, so its nuisance is shared and survives averaging.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
world = rng.normal(size=(2000, 32))
def world_share(rep):
    # fraction of representation variance explained by the world (projection onto world, per-dim R^2 mean)
    num = (rep * world).sum(0) / (world * world).sum(0)
    pred = world * num
    return 1 - ((rep - pred)**2).sum() / (rep**2).sum()
n1, n2 = rng.normal(size=world.shape), rng.normal(size=world.shape)
good_rep = world + (n1 + n2) / 2                    # two independent observations
bad_rep = world + n1 + 0.3*rng.normal(size=world.shape) / 2  # corruption of view 1: nuisance n1 fully shared
print(f"world share, good views: {world_share(good_rep):.3f}")
print(f"world share, bad views:  {world_share(bad_rep):.3f}")
assert world_share(good_rep) > world_share(bad_rep)
print("OK: good views wash out modality noise; bad views bake it into the representation")

## Scaling with data (paper's reported ladder, 33M JEPA)

Flickr captions only: 0.034. Texts of 50M web images: 0.125 — level with much larger language models. Texts of 940M web images (109M JEPA): 0.184.

In [ ]:
flickr, web50m, web940m = 0.034, 0.125, 0.184
print(f"Flickr -> 50M web images: {web50m/flickr:.2f}x alignment")
print(f"50M -> 940M web images: {web940m/web50m:.2f}x alignment")
assert web940m > web50m > flickr